# UWF Zeek GNN — blocked, grouped cross-validation (end to end)

One notebook from the raw UWF parquet files to cross-validated GraphSAGE results.

**What is kept from Sai's pipeline** (`Threat Detection - Data Prep.ipynb`, `1_Threat_Detection_GNN_training.ipynb`, `4_Threat_Detection_Tactic_Classification.ipynb`):
- label cleaning (`binary_target`, `tactic_target`, `mitre_target`, `technique_context`) — same SQL
- year split (2021–22 train, 2024 validation, 2025 test) — kept as the `data_split` column and as the reference run
- 5-minute edges with the same 10 traffic columns and the label lists `tactic_targets` / `mitre_targets`
- edge features (log1p + protocol ratios), 1-hour context graphs, 6 node features, `EdgeGraphSAGE` and its training recipe

**What is new**
- a `capture` column (the UWF dataset each log came from), added to the edge key so concurrent captures never merge
- blocks: continuous stretches of attack activity (split at gaps over 65 minutes) and normal-only days
- 5-fold `StratifiedGroupKFold` over blocks, stratified on year + dominant technique, repeated 3 times
- per-fold roles (train / val / test / purged) with a 65-minute purge around test and validation blocks
- leave-one-year-out as a robustness check

Sections: 0 config · 1 raw files · 2 ingest · 3 labels · 4 edges · 5 blocks and folds · 6 roles · 7 graphs · 8 models · 9 runs · 10 results

## 0. Configuration

In [7]:
import importlib.util, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
for package, module in [("duckdb", "duckdb"), ("torch-geometric", "torch_geometric")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import gc, json, os, re, shutil, time, urllib.parse
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import requests

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DRIVE_BASE = Path("/content/drive/Shareddrives/298B") if IN_COLAB else Path("./uwf_work")

CFG = {
    # raw data: <RAW_DIR>/<capture>/<week>/part-*.parquet (the layout of uwf_raw_backup on Drive)
    "RAW_DIR": str(DRIVE_BASE / "uwf_raw_backup"),
    "DOWNLOAD_IF_MISSING": True,
    "UWF_URL": "https://datasets.uwf.edu/data/",
    "CAPTURES": ["UWF-ZeekData22", "UWF-ZeekDataFall22", "UWF-ZeekData24",
                 "UWF-ZeekDataFall24-2", "UWF-ZeekDataSum25-1", "UWF-ZeekDataSum25-2"],
    "EXPECTED_FILES": 50,
    # outputs (kept on Drive so a disconnected runtime can resume)
    "OUT_DIR": str(DRIVE_BASE / "blocked_cv"),
    "WORK_DIR": "/content/work" if IN_COLAB else "./uwf_work/local",
    "DUCKDB_MEMORY": "8GB",
    "REBUILD_EDGES": False,
    # graphs
    "CONTEXT_MINUTES": 60,
    "GRAPH_SCOPE": "capture",          # "global" = Sai's graphs (all captures in one window); "capture" = one graph per capture
    "REBUILD_GRAPHS": False,
    # blocks, folds, purge
    "PURGE_MINUTES": 65, # 60-minute context + one 5-minute window
    "MAX_BLOCK_HOURS": None,
    "N_FOLDS": 5,
    "N_REPEATS": 3,
    "MIN_BLOCKS_PER_STRATUM": 5,
    "SEED": 42,
    # what to run
    "TASKS": ["binary"],              # add "tactic" and/or "technique" for the multi-label heads
    "RUN_REFERENCE": True,            # Sai's year split, same code path
    "RUN_CV": True,
    "REPEATS_TO_RUN": [0],            # start with one repeat; [0, 1, 2] for the full protocol
    "RUN_LOYO": True,
    "EPOCHS": 20,
    "PATIENCE": 5,
}
# Optional overrides, e.g. for a quick test run: UWF_CV_OVERRIDES='{"EPOCHS": 2}'
CFG.update(json.loads(os.environ.get("UWF_CV_OVERRIDES", "{}")))

RAW_DIR, OUT_DIR, WORK_DIR = Path(CFG["RAW_DIR"]), Path(CFG["OUT_DIR"]), Path(CFG["WORK_DIR"])
RESULT_DIR = OUT_DIR / f"results_{CFG['GRAPH_SCOPE']}_max{CFG['MAX_BLOCK_HOURS'] or 'none'}h"
for path in [OUT_DIR, WORK_DIR, RESULT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

CONTEXT = pd.Timedelta(minutes=CFG["CONTEXT_MINUTES"])
PURGE = pd.Timedelta(minutes=CFG["PURGE_MINUTES"])

# Sai's notebook outputs, used as parity checks when the full dataset is loaded
SAI = {
    "raw_rows": 27_111_594,
    "label_binary": {"unknown": 18_562_468, "False": 4_274_703, "True": 4_241_370, "Duplicate": 33_053},
    "binary_rows": {0: 13_556_302, 1: 13_522_239},
    "split_rows": {"train": 19_259_740, "validation": 4_800_145, "test": 3_018_656},
    "split_edges": {"train": 150_914, "validation": 295_052, "test": 232_088},
    "split_attack_edges": {"train": 779, "validation": 52_913, "test": 958},
}
print("Colab:", IN_COLAB, "| raw:", RAW_DIR, "| out:", OUT_DIR)

Colab: True | raw: /content/drive/Shareddrives/298B/uwf_raw_backup | out: /content/drive/Shareddrives/298B/blocked_cv


## 1. Raw files
Uses the Drive copy in `uwf_raw_backup` when present, otherwise downloads the 50 parquet files from datasets.uwf.edu.
The capture name is the first folder under `RAW_DIR`; Sai's combined file lost it, which is why we start from the raw files.

In [ ]:
def list_raw_files():
    rows = []
    for path in sorted(RAW_DIR.glob("*/*/part-*.parquet")):
        rows.append({"capture": path.parts[-3], "week": path.parts[-2], "path": str(path)})
    return pd.DataFrame(rows, columns=["capture", "week", "path"])


def server_links(url):
    html = requests.get(url, timeout=60).text
    return re.findall(r'<a href="([^"?]+)"', html)


def download_raw_files():
    for capture in CFG["CAPTURES"]:
        parquet_url = f"{CFG['UWF_URL']}{capture}/parquet/"
        for week in server_links(parquet_url):
            if not week.endswith("/") or week.startswith(("..", "_")):
                continue
            week_dir = RAW_DIR / capture / re.sub(r"[^0-9A-Za-z]+", "_", urllib.parse.unquote(week.strip("/")))
            week_dir.mkdir(parents=True, exist_ok=True)
            for name in server_links(parquet_url + week):
                if name.startswith("part-") and name.endswith(".parquet") and not (week_dir / name).exists():
                    with requests.get(parquet_url + week + name, stream=True, timeout=120) as response:
                        response.raise_for_status()
                        with open(week_dir / (name + ".part"), "wb") as handle:
                            for chunk in response.iter_content(1 << 20):
                                handle.write(chunk)
                    (week_dir / (name + ".part")).rename(week_dir / name)
                    print("downloaded", capture, week_dir.name, name)


raw_files = list_raw_files()
if raw_files.empty and CFG["DOWNLOAD_IF_MISSING"]:
    download_raw_files()
    raw_files = list_raw_files()

print(raw_files.groupby("capture").size().rename("files"))
print("Total files:", len(raw_files))
if CFG["EXPECTED_FILES"] and len(raw_files) != CFG["EXPECTED_FILES"]:
    print(f"WARNING: expected {CFG['EXPECTED_FILES']} files, found {len(raw_files)}")

downloaded UWF-ZeekData22 2021_12_12_2021_12_19 part-00000-7c2e9adb-5430-4792-a42b-10ff5bbd46e8-c000.snappy.parquet
downloaded UWF-ZeekData22 2021_12_19_2021_12_26 part-00000-3f86626a-1225-47f9-a5a2-0170b737e404-c000.snappy.parquet
downloaded UWF-ZeekData22 2021_12_26_2022_01_02 part-00000-b1a9fc13-8068-4a5d-91b2-871438709e81-c000.snappy.parquet
downloaded UWF-ZeekData22 2022_01_02_2022_01_09 part-00000-26e9208e-7819-451b-b23f-2e47f6d1e834-c000.snappy.parquet
downloaded UWF-ZeekData22 2022_01_09_2022_01_16 part-00000-36240b61-b84f-4164-a873-d7973e652780-c000.snappy.parquet
downloaded UWF-ZeekData22 2022_01_16_2022_01_23 part-00000-cbf26680-106d-40e7-8278-60520afdbb0e-c000.snappy.parquet
downloaded UWF-ZeekData22 2022_02_06_2022_02_13 part-00000-df678a79-4a73-452b-8e72-d624b2732f17-c000.snappy.parquet
downloaded UWF-ZeekData22 2022_02_13_2022_02_20 part-00000-1da06990-329c-4e38-913a-0f0aa39b388d-c000.snappy.parquet
downloaded UWF-ZeekDataFall22 2021_12_12_2021_12_19 part-00000-d512890f-

## 2. Ingest into DuckDB
Same harmonisation as Sai's merge: the four columns missing in older files (`label_technique`, `label_cve`, `label_binary`, `vlan`) become the string `'unknown'`, and booleans become `'True'` / `'False'`.

In [ ]:
con = duckdb.connect(str(WORK_DIR / "uwf_blocked_cv.duckdb"))
con.execute(f"SET memory_limit = '{CFG['DUCKDB_MEMORY']}'")
con.execute(f"SET temp_directory = '{WORK_DIR / 'duckdb_temp'}'")
con.execute("SET preserve_insertion_order = false")

RAW_COLUMNS = [
    ("datetime", "TIMESTAMP"), ("src_ip_zeek", "VARCHAR"), ("dest_ip_zeek", "VARCHAR"),
    ("dest_port_zeek", "BIGINT"), ("proto", "VARCHAR"), ("service", "VARCHAR"), ("conn_state", "VARCHAR"),
    ("duration", "DOUBLE"), ("orig_bytes", "DOUBLE"), ("resp_bytes", "DOUBLE"),
    ("orig_pkts", "DOUBLE"), ("resp_pkts", "DOUBLE"), ("label_tactic", "VARCHAR"),
]
STRING_LABELS = ["label_technique", "label_cve", "label_binary", "vlan"]


def select_list(path):
    types = {row[0]: row[1] for row in con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}')").fetchall()}
    parts = []
    for name, sql_type in RAW_COLUMNS:
        parts.append(f"TRY_CAST({name} AS {sql_type}) AS {name}" if name in types else f"CAST(NULL AS {sql_type}) AS {name}")
    for name in STRING_LABELS:
        if name not in types:
            parts.append(f"'unknown' AS {name}")
        elif types[name] == "BOOLEAN":
            parts.append(f"CASE WHEN {name} THEN 'True' WHEN NOT {name} THEN 'False' ELSE 'unknown' END AS {name}")
        else:
            parts.append(f"COALESCE(NULLIF(CAST({name} AS VARCHAR), 'nan'), 'unknown') AS {name}")
    return ", ".join(parts)


have_raw = con.execute("SELECT COUNT(*) FROM information_schema.tables WHERE table_name = 'raw_logs'").fetchone()[0]
if not have_raw or CFG["REBUILD_EDGES"]:
    con.execute("DROP TABLE IF EXISTS raw_logs")
    for index, row in enumerate(raw_files.itertuples()):
        path = row.path.replace("'", "''")
        query = f"SELECT {select_list(path)}, '{row.capture}' AS capture, '{row.week}' AS source_period FROM read_parquet('{path}')"
        con.execute(f"CREATE TABLE raw_logs AS {query}" if index == 0 else f"INSERT INTO raw_logs {query}")
        if (index + 1) % 10 == 0 or index + 1 == len(raw_files):
            print(f"  {index + 1}/{len(raw_files)} files")

raw_rows = con.execute("SELECT COUNT(*) FROM raw_logs").fetchone()[0]
FULL_DATA = raw_rows == SAI["raw_rows"]
print(f"raw_logs: {raw_rows:,} rows | full dataset: {FULL_DATA}")
con.execute("""
SELECT capture, COUNT(*) AS logs, MIN(datetime) AS first_log, MAX(datetime) AS last_log
FROM raw_logs GROUP BY capture ORDER BY first_log
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  30/50 files
  40/50 files


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  50/50 files
raw_logs: 27,111,594 rows | full dataset: True


,capture,logs,first_log,last_log
0,UWF-ZeekData22,18562468,2021-12-17 07:00:45.213000,2022-02-19 19:16:36.163000
1,UWF-ZeekDataFall22,700340,2021-12-17 13:00:45.251239,2022-10-23 22:07:45.793000
2,UWF-ZeekData24,1916757,2024-02-28 04:00:37.805000,2024-11-05 23:59:48.335452
3,UWF-ZeekDataFall24-2,2909841,2024-09-21 21:06:52.207000,2024-12-09 23:59:51.394266
4,UWF-ZeekDataSum25-1,1153509,2025-05-23 00:00:00.895212,2025-07-31 14:56:10.005000
5,UWF-ZeekDataSum25-2,1868679,2025-05-23 04:37:17.535901,2025-06-09 20:12:33.072000


In [ ]:
## parity with Sai's notebook: label_binary values
label_binary = dict(con.execute("SELECT label_binary, COUNT(*) FROM raw_logs GROUP BY 1").fetchall())
print(label_binary)
if FULL_DATA:
    assert label_binary == SAI["label_binary"], "label_binary counts differ from Sai's notebook"
    print("[PASS] label_binary counts match Sai's notebook")

{'Duplicate': 33053, 'False': 4274703, 'unknown': 18562468, 'True': 4241370}
[PASS] label_binary counts match Sai's notebook


## 3. Labels (Sai's `model_labels` view, unchanged) and the year split

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW model_labels AS

WITH binary_labels AS (
    SELECT
        *,
        CASE
            WHEN label_binary = 'True' THEN 1
            WHEN label_binary = 'False' THEN 0
            WHEN label_binary = 'unknown'
                 AND LOWER(TRIM(label_tactic)) = 'none'
                THEN 0
            WHEN label_binary = 'unknown'
                 AND label_tactic IS NOT NULL
                 AND LOWER(TRIM(label_tactic)) NOT IN ('none', 'unknown', '')
                THEN 1
            ELSE NULL
        END AS binary_target
    FROM raw_logs
    WHERE label_binary != 'Duplicate'
),

clean_targets AS (
    SELECT
        *,
        CASE
            WHEN binary_target = 0 THEN 'none'
            WHEN binary_target = 1
                 AND label_tactic IS NOT NULL
                 AND LOWER(TRIM(label_tactic)) NOT IN ('none', 'unknown', '')
                THEN TRIM(label_tactic)
            ELSE NULL
        END AS tactic_target,
        CASE
            WHEN binary_target = 1
                 AND regexp_full_match(TRIM(label_technique), 'T[0-9]{4}([.][0-9]{3})?')
                THEN TRIM(label_technique)
            ELSE NULL
        END AS mitre_target
    FROM binary_labels
)

SELECT
    *,
    CASE
        WHEN binary_target = 0 THEN 'none'
        WHEN mitre_target IS NOT NULL THEN mitre_target
        WHEN binary_target = 1 AND tactic_target IS NOT NULL THEN tactic_target || '_unspecified'
        ELSE NULL
    END AS technique_context
FROM clean_targets
""")

## Sai's year split, kept as a column (reference run and leave-one-year-out use it)
con.execute("""
CREATE OR REPLACE VIEW split_logs AS
SELECT
    *,
    CASE
        WHEN YEAR(datetime) IN (2021, 2022) THEN 'train'
        WHEN YEAR(datetime) = 2024 THEN 'validation'
        WHEN YEAR(datetime) = 2025 THEN 'test'
    END AS data_split
FROM model_labels
WHERE binary_target IS NOT NULL
  AND YEAR(datetime) IN (2021, 2022, 2024, 2025)
""")

binary_rows = dict(con.execute("SELECT binary_target, COUNT(*) FROM model_labels GROUP BY 1").fetchall())
split_rows = dict(con.execute("SELECT data_split, COUNT(*) FROM split_logs GROUP BY 1").fetchall())
print("binary_target:", binary_rows)
print("data_split:", split_rows)
if FULL_DATA:
    assert {k: binary_rows.get(k) for k in (0, 1)} == SAI["binary_rows"]
    assert split_rows == SAI["split_rows"]
    print("[PASS] labels and year split match Sai's notebook")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

binary_target: {1: 13522239, 0: 13556302}
data_split: {'test': 3018656, 'train': 19259740, 'validation': 4800145}
[PASS] labels and year split match Sai's notebook


## 4. Five-minute edges
Sai's two month-by-month aggregations (traffic features, then label lists) in one query, with `capture` added to the key.

In [9]:
EDGE_FILE = OUT_DIR / "edges_5min_enriched_capture.parquet"

if CFG["REBUILD_EDGES"] or not EDGE_FILE.exists():
    months = con.execute("""
        SELECT DISTINCT YEAR(datetime) AS year, MONTH(datetime) AS month
        FROM split_logs ORDER BY year, month
    """).fetchall()

    con.execute("DROP TABLE IF EXISTS edges_5min")
    for index, (year, month) in enumerate(months):
        print(f"Processing {year}-{month:02d}...")
        edge_query = f"""
        SELECT
            capture,
            data_split,
            TIME_BUCKET(INTERVAL '5 minutes', datetime) AS window_start,
            src_ip_zeek,
            dest_ip_zeek,
            COUNT(*) AS connection_count,
            SUM(CAST(COALESCE(duration, 0) AS DOUBLE)) AS total_duration,
            MAX(COALESCE(duration, 0)) AS maximum_duration,
            SUM(CAST(COALESCE(orig_bytes, 0) AS DOUBLE)) AS total_orig_bytes,
            SUM(CAST(COALESCE(resp_bytes, 0) AS DOUBLE)) AS total_resp_bytes,
            SUM(CAST(COALESCE(orig_pkts, 0) AS DOUBLE)) AS total_orig_packets,
            SUM(CAST(COALESCE(resp_pkts, 0) AS DOUBLE)) AS total_resp_packets,
            SUM(CASE WHEN proto = 'tcp' THEN 1 ELSE 0 END) AS tcp_count,
            SUM(CASE WHEN proto = 'udp' THEN 1 ELSE 0 END) AS udp_count,
            SUM(CASE WHEN proto = 'icmp' THEN 1 ELSE 0 END) AS icmp_count,
            MIN(binary_target) AS minimum_binary_target,
            MAX(binary_target) AS binary_target,
            LIST(DISTINCT tactic_target ORDER BY tactic_target)
                FILTER (WHERE binary_target = 1 AND tactic_target IS NOT NULL AND tactic_target != 'none')
                AS tactic_targets,
            LIST(DISTINCT mitre_target ORDER BY mitre_target)
                FILTER (WHERE binary_target = 1 AND mitre_target IS NOT NULL)
                AS mitre_targets
        FROM split_logs
        WHERE YEAR(datetime) = {year} AND MONTH(datetime) = {month}
        GROUP BY capture, data_split, window_start, src_ip_zeek, dest_ip_zeek
        """
        con.execute(f"CREATE TABLE edges_5min AS {edge_query}" if index == 0 else f"INSERT INTO edges_5min {edge_query}")
        con.execute("CHECKPOINT")
        gc.collect()

    con.execute(f"COPY (SELECT * FROM edges_5min ORDER BY window_start, capture, src_ip_zeek, dest_ip_zeek) TO '{EDGE_FILE}' (FORMAT PARQUET, COMPRESSION ZSTD)")
    print("Saved:", EDGE_FILE)

edges = pd.read_parquet(EDGE_FILE)
edges = edges.sort_values(["window_start", "capture", "src_ip_zeek", "dest_ip_zeek"], kind="mergesort").reset_index(drop=True)
print(f"edges: {len(edges):,} | attack edges: {int(edges.binary_target.sum()):,}")

edges: 726,307 | attack edges: 54,650


In [11]:
## checks: no mixed edges, and Sai's edge counts when the capture is left out of the key
if "FULL_DATA" not in globals():
    # Fallback definition: check if the loaded edges match the full expected dataset shape
    FULL_DATA = len(edges) == 726307

mixed = int((edges.minimum_binary_target != edges.binary_target).sum())
sai_key = edges.groupby(["data_split", "window_start", "src_ip_zeek", "dest_ip_zeek"]).binary_target.max().reset_index()
sai_edges = sai_key.groupby("data_split").size().to_dict()
sai_attacks = sai_key.groupby("data_split").binary_target.sum().astype(int).to_dict()
merged = len(edges) - len(sai_key)

print("mixed attack/normal edges:", mixed)
print("edges with Sai's key:", sai_edges, "| attack:", sai_attacks)
print(f"edges that Sai's key merged across concurrent captures: {merged:,}")
print(edges.groupby(["data_split", "capture"]).agg(edges=("binary_target", "size"), attack_edges=("binary_target", "sum")))
if FULL_DATA:
    assert mixed == 0
    assert sai_edges == SAI["split_edges"] and sai_attacks == SAI["split_attack_edges"]
    print("[PASS] edge counts match Sai's notebook")

mixed attack/normal edges: 0
edges with Sai's key: {'test': 232088, 'train': 150914, 'validation': 295052} | attack: {'test': 958, 'train': 779, 'validation': 52913}
edges that Sai's key merged across concurrent captures: 48,253
                                  edges  attack_edges
data_split capture                                   
test       UWF-ZeekDataSum25-1   179554           675
           UWF-ZeekDataSum25-2    73215           283
train      UWF-ZeekData22        145608           340
           UWF-ZeekDataFall22     32878           439
validation UWF-ZeekData24        109011         51771
           UWF-ZeekDataFall24-2  186041          1142
[PASS] edge counts match Sai's notebook


In [12]:
key = ["data_split", "window_start", "src_ip_zeek", "dest_ip_zeek"]
shared = edges[edges.duplicated(key, keep=False)]
per_key = shared.groupby(key).agg(
    captures=("capture", lambda s: " + ".join(sorted(s))),
    distinct_counts=("connection_count", "nunique"),
    distinct_bytes=("total_orig_bytes", "nunique"),
)
print(f"shared IP pairs/windows: {len(per_key):,}")
print(per_key.captures.value_counts())
print(f"identical connection counts: {(per_key.distinct_counts == 1).mean():.1%}")
print(f"identical bytes:             {(per_key.distinct_bytes == 1).mean():.1%}")

shared IP pairs/windows: 48,253
captures
UWF-ZeekData22 + UWF-ZeekDataFall22          27572
UWF-ZeekDataSum25-1 + UWF-ZeekDataSum25-2    20681
Name: count, dtype: int64
identical connection counts: 1.9%
identical bytes:             8.7%


In [13]:
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

path = "/content/drive/Shareddrives/298B/blocked_cv/edges_5min_enriched_capture.parquet"
df = pd.read_parquet(path)

print(df.shape)                      # rows, columns
print(df.dtypes)                     # column names and types
display(df.head(10))                 # first 10 edges
display(df[df.binary_target == 1].sample(10, random_state=0))   # 10 random attack edges
display(df.describe().T)             # numeric summary

(726307, 19)
capture                          object
data_split                       object
window_start             datetime64[us]
src_ip_zeek                      object
dest_ip_zeek                     object
connection_count                  int64
total_duration                  float64
maximum_duration                float64
total_orig_bytes                float64
total_resp_bytes                float64
total_orig_packets              float64
total_resp_packets              float64
tcp_count                       float64
udp_count                       float64
icmp_count                      float64
minimum_binary_target             int32
binary_target                     int32
tactic_targets                   object
mitre_targets                    object
dtype: object


,capture,data_split,window_start,src_ip_zeek,dest_ip_zeek,connection_count,total_duration,maximum_duration,total_orig_bytes,total_resp_bytes,total_orig_packets,total_resp_packets,tcp_count,udp_count,icmp_count,minimum_binary_target,binary_target,tactic_targets,mitre_targets
0,UWF-ZeekData22,train,2021-12-17 07:00:00,0.0.0.0,255.255.255.255,2,3.463008e+06,1.731504e+06,121572814.0,0.0,405242.0,0.0,0.0,2.0,0.0,0,0,None,None
1,UWF-ZeekData22,train,2021-12-17 07:00:00,127.0.0.1,127.0.0.1,4,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0,0,None,None
2,UWF-ZeekData22,train,2021-12-17 07:00:00,143.88.0.2,143.88.255.10,2,3.246063e+06,1.623032e+06,363553120.0,363553120.0,6492020.0,6492020.0,0.0,0.0,2.0,0,0,None,None
3,UWF-ZeekData22,train,2021-12-17 07:00:00,143.88.0.2,192.33.4.12,2,0.000000e+00,0.000000e+00,0.0,0.0,2.0,0.0,0.0,2.0,0.0,0,0,None,None
4,UWF-ZeekData22,train,2021-12-17 07:00:00,143.88.0.2,192.36.148.17,2,0.000000e+00,0.000000e+00,0.0,0.0,2.0,0.0,0.0,2.0,0.0,0,0,None,None
5,UWF-ZeekData22,train,2021-12-17 07:00:00,143.88.0.2,198.97.190.53,2,0.000000e+00,0.000000e+00,0.0,0.0,2.0,0.0,0.0,2.0,0.0,0,0,None,None
6,UWF-ZeekData22,train,2021-12-17 07:00:00,143.88.0.2,199.9.14.201,2,0.000000e+00,0.000000e+00,0.0,0.0,2.0,0.0,0.0,2.0,0.0,0,0,None,None
7,UWF-ZeekData22,train,2021-12-17 07:00:00,143.88.1.50,143.88.1.255,4,2.402092e+01,8.006439e+00,2450.0,0.0,18.0,0.0,0.0,4.0,0.0,0,0,None,None
8,UWF-ZeekData22,train,2021-12-17 07:00:00,143.88.1.50,143.88.255.10,2,3.246063e+06,1.623032e+06,181776560.0,181776560.0,3246010.0,3246010.0,0.0,0.0,2.0,0,0,None,None
9,UWF-ZeekData22,train,2021-12-17 07:00:00,143.88.1.51,143.88.1.1,49,2.401636e+02,5.005035e+00,8718.0,2328.0,194.0,194.0,0.0,49.0,0.0,0,0,None,None


,capture,data_split,window_start,src_ip_zeek,dest_ip_zeek,connection_count,total_duration,maximum_duration,total_orig_bytes,total_resp_bytes,total_orig_packets,total_resp_packets,tcp_count,udp_count,icmp_count,minimum_binary_target,binary_target,tactic_targets,mitre_targets
208275,UWF-ZeekData24,validation,2024-03-18 21:50:00,143.88.4.11,143.88.5.7,2,0.001866,0.001139,0.0,0.0,6.0,0.0,2.0,0.0,0.0,1,1,[Reconnaissance],[T1595]
218054,UWF-ZeekData24,validation,2024-03-22 01:15:00,143.88.7.11,143.88.8.10,1,0.097006,0.097006,619.0,201.0,24.0,16.0,1.0,0.0,0.0,1,1,[Initial Access],[T1190]
185289,UWF-ZeekData24,validation,2024-03-01 11:55:00,143.88.10.12,143.88.11.14,1,0.004060,0.004060,0.0,0.0,8.0,4.0,1.0,0.0,0.0,1,1,[Reconnaissance],[T1595]
725767,UWF-ZeekDataSum25-1,test,2025-07-16 03:05:00,143.88.1.13,143.88.1.11,37,0.256385,0.039292,42181.0,186516.0,317.0,209.0,37.0,0.0,0.0,1,1,[Reconnaissance],[T1595]
201581,UWF-ZeekData24,validation,2024-03-16 17:50:00,143.88.12.12,143.88.13.14,1,0.106301,0.106301,571.0,202.0,24.0,16.0,1.0,0.0,0.0,1,1,[Initial Access],[T1190]
182182,UWF-ZeekData24,validation,2024-02-29 10:20:00,143.88.4.11,143.88.5.18,2,0.001147,0.000690,0.0,0.0,6.0,0.0,2.0,0.0,0.0,1,1,[Reconnaissance],[T1595]
205515,UWF-ZeekData24,validation,2024-03-18 00:20:00,143.88.5.14,143.88.6.1,2,0.001930,0.001226,0.0,0.0,6.0,0.0,2.0,0.0,0.0,1,1,[Reconnaissance],[T1595]
198323,UWF-ZeekData24,validation,2024-03-15 16:45:00,143.88.1.18,143.88.2.21,1,0.002203,0.002203,0.0,0.0,8.0,4.0,1.0,0.0,0.0,1,1,[Reconnaissance],[T1595]
215421,UWF-ZeekData24,validation,2024-03-21 04:50:00,143.88.4.11,143.88.5.6,2,0.002011,0.001095,0.0,0.0,6.0,0.0,2.0,0.0,0.0,1,1,[Reconnaissance],[T1595]
212493,UWF-ZeekData24,validation,2024-03-20 06:10:00,143.88.15.10,143.88.1.21,1,0.100152,0.100152,618.0,202.0,24.0,16.0,1.0,0.0,0.0,1,1,[Initial Access],[T1190]


,count,mean,min,25%,50%,75%,max,std
window_start,726307,2024-04-26 01:32:53.354794,2021-12-17 07:00:00,2024-02-29 05:15:00,2024-12-01 15:10:00,2025-05-26 08:50:00,2025-07-31 14:55:00,NaN
connection_count,726307.0,37.2825,1.0,1.0,2.0,9.0,742656.0,2445.458178
total_duration,726307.0,56.801735,0.0,0.0,0.00121,0.221191,3463007.623432,10467.558016
maximum_duration,726307.0,24.315249,0.0,0.0,0.000372,0.099673,2561894.922662,6078.570309
total_orig_bytes,726307.0,5135287.406254,0.0,0.0,212.0,1428.0,2575967594752.0,3213296418.744362
total_resp_bytes,726307.0,67924.508179,0.0,0.0,0.0,0.0,3910335612.0,9046008.243491
total_orig_packets,726307.0,193.403455,0.0,2.0,6.0,30.0,6492020.0,15472.934611
total_resp_packets,726307.0,143.014455,0.0,0.0,0.0,1.0,6492020.0,12485.30192
tcp_count,726307.0,18.370558,0.0,0.0,0.0,0.0,742144.0,2437.119138
udp_count,726307.0,18.825864,0.0,1.0,2.0,8.0,31744.0,157.328365


## 5. Blocks and folds
- **attack block**: a continuous stretch of 5-minute windows with attack activity (any capture), split where the gap exceeds 65 minutes; it holds every edge in that stretch, attack and normal
- **normal block**: the remaining windows of one calendar day
- **stratum**: year group + the block's most common technique (rare strata merged into `other`)
- folds: `StratifiedGroupKFold` on edges with `groups = block_id`, repeated with different seeds

In [14]:
from sklearn.model_selection import StratifiedGroupKFold

YEAR_GROUP = {"train": "2021-22", "validation": "2024", "test": "2025"}
edges["year_group"] = edges.data_split.map(YEAR_GROUP)


def label_list(value):
    if value is None:
        return []
    if isinstance(value, float) and np.isnan(value):
        return []
    return [str(x) for x in value]


edges["mitre_list"] = edges.mitre_targets.map(label_list)
edges["tactic_list"] = edges.tactic_targets.map(label_list)
edges["technique"] = np.where(
    edges.binary_target == 1,
    [m[0] if m else (t[0] + "_unspecified" if t else "unlabelled") for m, t in zip(edges.mitre_list, edges.tactic_list)],
    "normal",
)

## attack blocks: continuous stretches of attack windows
attack_times = np.sort(edges.loc[edges.binary_target == 1, "window_start"].unique())
span_id = np.cumsum(np.r_[True, np.diff(attack_times) > PURGE.to_timedelta64()])
spans = pd.DataFrame({"window_start": attack_times, "span": span_id})
spans = spans.groupby("span").window_start.agg(["min", "max"])

window_block = pd.Series(
    "N_" + pd.Series(np.unique(edges.window_start)).dt.strftime("%Y-%m-%d").to_numpy(),
    index=np.unique(edges.window_start),
)
## long stretches are cut into chunks of at most MAX_BLOCK_HOURS, so one campaign cannot fill a fold
max_block = pd.Timedelta(hours=CFG["MAX_BLOCK_HOURS"]) if CFG["MAX_BLOCK_HOURS"] else None
for span, (start, end) in spans.iterrows():
    windows = window_block.loc[start:end].index
    chunks = ((windows - start) // max_block).astype(int) if max_block is not None else np.zeros(len(windows), dtype=int)
    window_block.loc[start:end] = [f"A_{span:04d}_{chunk:03d}" for chunk in chunks]
edges["block_id"] = edges.window_start.map(window_block)

## biggest blocks; attack-only blocks have no normal edges, so they test recall but not false positives
block_size = edges.groupby("block_id").agg(
    start=("window_start", "min"), end=("window_start", "max"),
    edges=("binary_target", "size"), attack_edges=("binary_target", "sum"),
    captures=("capture", lambda s: ", ".join(sorted(s.unique()))),
)
block_size["hours"] = ((block_size.end - block_size.start).dt.total_seconds() / 3600).round(1)
block_size["attack_only"] = block_size.edges == block_size.attack_edges
print(f"attack-only blocks: {block_size.attack_only.sum()} holding "
      f"{block_size.loc[block_size.attack_only, 'attack_edges'].sum():,} attack edges")
print(block_size.sort_values("attack_edges", ascending=False).head(10).to_string())

## stratum per block: year group + most common technique of its attack edges
attack_rows = edges[edges.binary_target == 1]
block_technique = attack_rows.groupby("block_id").technique.agg(lambda s: s.value_counts().index[0])
block_year = edges.groupby("block_id").year_group.first()
blocks = pd.DataFrame({"year_group": block_year})
blocks["technique"] = block_technique.reindex(blocks.index).fillna("normal")
blocks["stratum"] = blocks.year_group + "|" + blocks.technique
small = blocks.stratum.map(blocks.stratum.value_counts()) < CFG["MIN_BLOCKS_PER_STRATUM"]
blocks.loc[small & (blocks.technique != "normal"), "stratum"] = blocks.year_group + "|other_attack"
small = blocks.stratum.map(blocks.stratum.value_counts()) < CFG["MIN_BLOCKS_PER_STRATUM"]
blocks.loc[small, "stratum"] = np.where(blocks.loc[small, "technique"] == "normal", "normal", "other_attack")
edges["stratum"] = edges.block_id.map(blocks.stratum)

print(f"blocks: {len(blocks):,} ({(blocks.technique != 'normal').sum():,} attack, {(blocks.technique == 'normal').sum():,} normal)")
print(blocks.stratum.value_counts())

attack-only blocks: 83 holding 54,331 attack edges
                         start                 end  edges  attack_edges                                  captures  hours  attack_only
block_id                                                                                                                             
A_0052_000 2024-03-13 21:00:00 2024-03-24 23:55:00  34566         34566                            UWF-ZeekData24  266.9         True
A_0051_000 2024-02-28 04:00:00 2024-03-03 23:55:00  14175         14175                            UWF-ZeekData24  115.9         True
A_0053_000 2024-03-27 00:00:00 2024-03-27 23:55:00   3030          3030                            UWF-ZeekData24   23.9         True
A_0063_000 2024-11-18 00:15:00 2024-11-18 00:30:00    765           765                      UWF-ZeekDataFall24-2    0.2         True
A_0068_000 2025-06-03 16:35:00 2025-06-03 16:50:00    411           267  UWF-ZeekDataSum25-1, UWF-ZeekDataSum25-2    0.2        False
A_0003_000 

In [15]:
for repeat in range(CFG["N_REPEATS"]):
    splitter = StratifiedGroupKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED"] + repeat)
    fold = np.full(len(edges), -1, dtype=np.int8)
    for k, (_, test_index) in enumerate(splitter.split(edges, edges.stratum, groups=edges.block_id)):
        fold[test_index] = k
    edges[f"fold_r{repeat}"] = fold
    assert (fold >= 0).all()
    assert edges.groupby("block_id")[f"fold_r{repeat}"].nunique().max() == 1, "a block was split across folds"

## balance of repeat 0
balance = edges.groupby("fold_r0").agg(
    edges=("binary_target", "size"),
    attack_edges=("binary_target", "sum"),
    blocks=("block_id", "nunique"),
    years=("year_group", lambda s: ", ".join(sorted(s.unique()))),
)
balance["attack_share"] = (balance.attack_edges / balance.edges).round(4)
balance

,edges,attack_edges,blocks,years,attack_share
fold_r0,,,,,
0,160602,446,35,"2021-22, 2024, 2025",0.0028
1,116727,336,26,"2021-22, 2024, 2025",0.0029
2,124845,1599,28,"2021-22, 2024, 2025",0.0128
3,203479,52117,41,"2021-22, 2024, 2025",0.2561
4,120654,152,28,"2021-22, 2024, 2025",0.0013


In [16]:
## technique coverage: in how many folds (repeat 0) each technique has test edges
coverage = (attack_rows.assign(fold=edges.loc[attack_rows.index, "fold_r0"])
            .groupby("technique").agg(attack_edges=("fold", "size"), folds=("fold", "nunique"), blocks=("block_id", "nunique"))
            .sort_values("attack_edges", ascending=False))
coverage["insufficient_data"] = coverage.blocks < 2
coverage

,attack_edges,folds,blocks,insufficient_data
technique,,,,
T1595,37156,5,30,False
T1078,5426,1,3,False
T1110,4837,1,3,False
T1190,4592,4,9,False
T1046,571,5,8,False
T1018,531,2,3,False
T1048,494,1,2,False
Reconnaissance_unspecified,315,4,9,False
T1590,287,5,13,False


In [17]:
FOLD_COLUMNS = ["capture", "window_start", "src_ip_zeek", "dest_ip_zeek", "data_split", "year_group",
                "block_id", "stratum", "technique"] + [f"fold_r{r}" for r in range(CFG["N_REPEATS"])]
edges[FOLD_COLUMNS].to_parquet(OUT_DIR / "edge_folds.parquet", index=False)
print("Saved:", OUT_DIR / "edge_folds.parquet")

Saved: /content/drive/Shareddrives/298B/blocked_cv/edge_folds.parquet


In [18]:
folds = pd.read_parquet("/content/drive/Shareddrives/298B/blocked_cv/edge_folds.parquet")
display(folds.head(10))
display(folds.groupby("fold_r0").agg(edges=("block_id", "size"), blocks=("block_id", "nunique")))

,capture,window_start,src_ip_zeek,dest_ip_zeek,data_split,year_group,block_id,stratum,technique,fold_r0,fold_r1,fold_r2
0,UWF-ZeekData22,2021-12-17 07:00:00,0.0.0.0,255.255.255.255,train,2021-22,N_2021-12-17,2021-22|normal,normal,3,2,3
1,UWF-ZeekData22,2021-12-17 07:00:00,127.0.0.1,127.0.0.1,train,2021-22,N_2021-12-17,2021-22|normal,normal,3,2,3
2,UWF-ZeekData22,2021-12-17 07:00:00,143.88.0.2,143.88.255.10,train,2021-22,N_2021-12-17,2021-22|normal,normal,3,2,3
3,UWF-ZeekData22,2021-12-17 07:00:00,143.88.0.2,192.33.4.12,train,2021-22,N_2021-12-17,2021-22|normal,normal,3,2,3
4,UWF-ZeekData22,2021-12-17 07:00:00,143.88.0.2,192.36.148.17,train,2021-22,N_2021-12-17,2021-22|normal,normal,3,2,3
5,UWF-ZeekData22,2021-12-17 07:00:00,143.88.0.2,198.97.190.53,train,2021-22,N_2021-12-17,2021-22|normal,normal,3,2,3
6,UWF-ZeekData22,2021-12-17 07:00:00,143.88.0.2,199.9.14.201,train,2021-22,N_2021-12-17,2021-22|normal,normal,3,2,3
7,UWF-ZeekData22,2021-12-17 07:00:00,143.88.1.50,143.88.1.255,train,2021-22,N_2021-12-17,2021-22|normal,normal,3,2,3
8,UWF-ZeekData22,2021-12-17 07:00:00,143.88.1.50,143.88.255.10,train,2021-22,N_2021-12-17,2021-22|normal,normal,3,2,3
9,UWF-ZeekData22,2021-12-17 07:00:00,143.88.1.51,143.88.1.1,train,2021-22,N_2021-12-17,2021-22|normal,normal,3,2,3


,edges,blocks
fold_r0,,
0,160602,35
1,116727,26
2,124845,28
3,203479,41
4,120654,28


In [19]:
b = edges.groupby("block_id").agg(
    start=("window_start", "min"), end=("window_start", "max"),
    edges=("binary_target", "size"), attack_edges=("binary_target", "sum"),
    captures=("capture", lambda s: ", ".join(sorted(s.unique()))),
    techniques=("technique", lambda s: ", ".join(s[s != "normal"].value_counts().index[:4])),
    fold=("fold_r0", "first"),
)
b["hours"] = ((b.end - b.start).dt.total_seconds() / 3600).round(1)
display(b.sort_values("attack_edges", ascending=False).head(15))
print(b.loc[b.attack_edges > 0, "hours"].describe())

,start,end,edges,attack_edges,captures,techniques,fold,hours
block_id,,,,,,,,
A_0052_000,2024-03-13 21:00:00,2024-03-24 23:55:00,34566,34566,UWF-ZeekData24,"T1595, T1078, T1110, T1190",3,266.9
A_0051_000,2024-02-28 04:00:00,2024-03-03 23:55:00,14175,14175,UWF-ZeekData24,"T1595, T1078, T1110, T1190",3,115.9
A_0053_000,2024-03-27 00:00:00,2024-03-27 23:55:00,3030,3030,UWF-ZeekData24,"T1595, T1078, T1110, T1190",3,23.9
A_0063_000,2024-11-18 00:15:00,2024-11-18 00:30:00,765,765,UWF-ZeekDataFall24-2,"T1018, T1046",2,0.2
A_0068_000,2025-06-03 16:35:00,2025-06-03 16:50:00,411,267,"UWF-ZeekDataSum25-1, UWF-ZeekDataSum25-2",T1046,1,0.2
A_0003_000,2022-02-10 17:45:00,2022-02-10 17:50:00,261,261,UWF-ZeekData22,Reconnaissance_unspecified,2,0.1
A_0019_000,2022-09-01 20:20:00,2022-09-04 20:05:00,261,261,UWF-ZeekDataFall22,"T1590, T1589, T1592",2,71.8
A_0062_000,2024-11-16 16:50:00,2024-11-16 16:55:00,257,257,UWF-ZeekDataFall24-2,T1548,2,0.1
A_0082_000,2025-07-22 17:05:00,2025-07-23 07:15:00,246,246,UWF-ZeekDataSum25-1,"T1595, T1210",3,14.2


count     90.000000
mean       6.564444
std       31.421099
min        0.000000
25%        0.000000
50%        0.150000
75%        1.175000
max      266.900000
Name: hours, dtype: float64


## 6. Roles per run
- `reference`: Sai's year split (train 2021–22, validation 2024, test 2025), no purge
- `cv`: test = fold k, validation = fold k+1, train = the rest; train and validation windows within 65 minutes of a test or validation window are purged
- `loyo`: test = one year group; validation = one fifth of the remaining blocks (stratified); same purge

In [20]:
WINDOW_TIMES = edges.window_start.to_numpy()


def near(times, other_times, gap):
    """True where a time lies within `gap` of any time in `other_times`."""
    other = np.unique(other_times)
    if len(other) == 0:
        return np.zeros(len(times), dtype=bool)
    position = np.searchsorted(other, times)
    before = other[np.clip(position - 1, 0, len(other) - 1)]
    after = other[np.clip(position, 0, len(other) - 1)]
    gap = gap.to_timedelta64()
    return (np.abs(times - before) <= gap) | (np.abs(after - times) <= gap)


def apply_purge(role):
    role = role.copy()
    test_times = WINDOW_TIMES[role == "test"]
    val_times = WINDOW_TIMES[role == "val"]
    role[(role == "train") & (near(WINDOW_TIMES, test_times, PURGE) | near(WINDOW_TIMES, val_times, PURGE))] = "purged"
    role[(role == "val") & near(WINDOW_TIMES, test_times, PURGE)] = "purged"
    return role


def roles_reference():
    return edges.data_split.map({"train": "train", "validation": "val", "test": "test"}).to_numpy()


def roles_cv(repeat, k):
    fold = edges[f"fold_r{repeat}"].to_numpy()
    role = np.where(fold == k, "test", np.where(fold == (k + 1) % CFG["N_FOLDS"], "val", "train")).astype(object)
    return apply_purge(role)


def roles_loyo(year_group):
    role = np.where(edges.year_group == year_group, "test", "train").astype(object)
    rest = np.flatnonzero(role == "train")
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=CFG["SEED"])
    _, val_index = next(splitter.split(rest, edges.stratum.to_numpy()[rest], groups=edges.block_id.to_numpy()[rest]))
    role[rest[val_index]] = "val"
    return apply_purge(role)


## leakage checks on every fold of repeat 0: no train window within the purge gap of a test window,
## and every edge is tested exactly once across the folds
tested = np.zeros(len(edges), dtype=int)
for k in range(CFG["N_FOLDS"]):
    role = roles_cv(0, k)
    tested += role == "test"
    assert not near(WINDOW_TIMES[role == "train"], WINDOW_TIMES[role == "test"], PURGE).any()
assert (tested == 1).all()
print("[PASS] purge gaps hold and every edge is tested exactly once")
print("cv repeat 0, fold 0 roles:\n", pd.Series(roles_cv(0, 0)).value_counts())

[PASS] purge gaps hold and every edge is tested exactly once
cv repeat 0, fold 0 roles:
 train     435407
test      160602
val       112076
purged     18222
Name: count, dtype: int64


## 7. Graphs
Sai's `build_graph`, run once for every 5-minute window. Context = the previous hour of edges (any role, features only);
targets = the window's edges. Edge features are stored unscaled; each run scales them with a scaler fitted on its own training edges.

In [21]:
import torch
from torch_geometric.data import Data

LOG_COLUMNS = ["connection_count", "total_duration", "maximum_duration", "total_orig_bytes",
               "total_resp_bytes", "total_orig_packets", "total_resp_packets"]


def create_edge_features(df):
    ## Sai's edge features: log1p of volumes + protocol ratios
    features = pd.DataFrame(index=df.index)
    for column in LOG_COLUMNS:
        features[f"log_{column}"] = np.log1p(df[column].fillna(0).clip(lower=0))
    denominator = df["connection_count"].clip(lower=1)
    for proto in ["tcp", "udp", "icmp"]:
        features[f"{proto}_ratio"] = df[f"{proto}_count"] / denominator
    return features


EDGE_FEATURES = create_edge_features(edges).to_numpy(np.float32)
GRAPH_FILE = OUT_DIR / f"graphs_{CFG['GRAPH_SCOPE']}.pt"


def make_graph(target_rows, context_rows, src, dst, volumes, window):
    nodes = np.concatenate([src[context_rows], dst[context_rows], src[target_rows], dst[target_rows]])
    codes, uniques = pd.factorize(nodes)
    n_context = len(context_rows)
    c_src, c_dst = codes[:n_context], codes[n_context:2 * n_context]
    t_src, t_dst = codes[2 * n_context:2 * n_context + len(target_rows)], codes[2 * n_context + len(target_rows):]

    ## node features from the context hour (Sai's 6 columns)
    node_features = np.zeros((len(uniques), 6), dtype=np.float64)
    if n_context:
        count, orig_b, resp_b, orig_p, resp_p = volumes[context_rows].T
        np.add.at(node_features[:, 0], c_src, count)
        np.add.at(node_features[:, 1], c_dst, count)
        np.add.at(node_features[:, 2], c_src, orig_b)
        np.add.at(node_features[:, 3], c_src, resp_b)
        np.add.at(node_features[:, 2], c_dst, resp_b)
        np.add.at(node_features[:, 3], c_dst, orig_b)
        np.add.at(node_features[:, 4], c_src, orig_p)
        np.add.at(node_features[:, 5], c_src, resp_p)
        np.add.at(node_features[:, 4], c_dst, resp_p)
        np.add.at(node_features[:, 5], c_dst, orig_p)

    pairs = np.unique(np.stack([c_src, c_dst], axis=1), axis=0) if n_context else np.zeros((0, 2), dtype=np.int64)
    graph = Data(
        x=torch.tensor(np.log1p(node_features), dtype=torch.float32),
        edge_index=torch.tensor(np.vstack([np.r_[pairs[:, 0], pairs[:, 1]], np.r_[pairs[:, 1], pairs[:, 0]]]), dtype=torch.long),
    )
    graph.target_edge_index = torch.tensor(np.vstack([t_src, t_dst]), dtype=torch.long)
    graph.target_edge_attr = torch.tensor(EDGE_FEATURES[target_rows], dtype=torch.float32)
    graph.y = torch.tensor(edges.binary_target.to_numpy()[target_rows], dtype=torch.float32)
    graph.row_ids = torch.tensor(target_rows, dtype=torch.long)
    graph.window_start = str(window)
    return graph


def build_graphs():
    src = edges.src_ip_zeek.to_numpy()
    dst = edges.dest_ip_zeek.to_numpy()
    volumes = edges[["connection_count", "total_orig_bytes", "total_resp_bytes",
                     "total_orig_packets", "total_resp_packets"]].to_numpy(np.float64)
    if CFG["GRAPH_SCOPE"] == "capture":
        parts = list(edges.groupby("capture").indices.values())
    else:
        parts = [np.arange(len(edges))]

    graphs = []
    for part in parts:
        part = part[np.argsort(WINDOW_TIMES[part], kind="stable")]
        times = WINDOW_TIMES[part]
        windows, starts = np.unique(times, return_index=True)
        ends = np.r_[starts[1:], len(part)]
        for j, window in enumerate(windows):
            first_context = np.searchsorted(times, window - CONTEXT.to_timedelta64(), side="left")
            graphs.append(make_graph(part[starts[j]:ends[j]], part[first_context:starts[j]], src, dst, volumes, window))
    return graphs


if CFG["REBUILD_GRAPHS"] or not GRAPH_FILE.exists():
    started = time.time()
    graphs = build_graphs()
    torch.save(graphs, GRAPH_FILE)
    print(f"built {len(graphs):,} graphs in {(time.time() - started) / 60:.1f} min")
else:
    graphs = torch.load(GRAPH_FILE, weights_only=False)

GRAPH_FIRST_ROW = np.array([int(g.row_ids[0]) for g in graphs])
print(f"graphs: {len(graphs):,} | target edges: {sum(len(g.y) for g in graphs):,} | "
      f"avg nodes: {np.mean([g.num_nodes for g in graphs]):.1f}")
assert sum(len(g.y) for g in graphs) == len(edges)

graphs: 33,607 | target edges: 726,307 | avg nodes: 66.5


## 8. Models and training (Sai's recipes)
- `binary`: `EdgeGraphSAGE`, BCE with `pos_weight = sqrt(normal / attack)`, AdamW 1e-3, model chosen on validation PR-AUC, threshold = best validation F1
- `tactic` / `technique`: `EdgeMultiLabelGraphSAGE` on attack edges with labels, BCE with per-class `pos_weight`, AdamW 3e-4, model chosen on validation micro PR-AUC, threshold grid 0.10–0.90 on micro F1

In [22]:
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (average_precision_score, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


class EdgeGraphSAGE(nn.Module):
    def __init__(self, num_outputs=1, node_features=6, edge_features=10, hidden_size=64, dropout=0.2):
        super().__init__()
        self.conv1 = SAGEConv(node_features, hidden_size)
        self.conv2 = SAGEConv(hidden_size, hidden_size)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2 + edge_features, hidden_size),
            nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_size, num_outputs),
        )
        self.dropout = dropout

    def forward(self, data):
        x = F.relu(self.conv1(data.x, data.edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, data.edge_index))
        source, destination = data.target_edge_index
        return self.classifier(torch.cat([x[source], x[destination], data.target_edge_attr], dim=1))


## label matrices for the multi-label heads
def multi_hot(lists):
    classes = sorted({label for values in lists for label in values})
    index = {label: i for i, label in enumerate(classes)}
    matrix = np.zeros((len(lists), len(classes)), dtype=np.float32)
    for row, values in enumerate(lists):
        for label in values:
            matrix[row, index[label]] = 1.0
    return classes, matrix


TASK_LABELS = {}
for task, column in [("tactic", "tactic_list"), ("technique", "mitre_list")]:
    classes, matrix = multi_hot(edges[column].tolist())
    usable = (edges.binary_target.to_numpy() == 1) & (matrix.sum(1) > 0)
    TASK_LABELS[task] = {"classes": classes, "Y": matrix, "usable": usable}
TASK_LABELS["binary"] = {"usable": np.ones(len(edges), dtype=bool)}
BINARY_Y = edges.binary_target.to_numpy().astype(np.float32)
GRAPH_USABLE = {task: np.array([TASK_LABELS[task]["usable"][g.row_ids.numpy()].any() for g in graphs])
                for task in ["binary", "tactic", "technique"]}


def binary_scores(y, p, threshold):
    predicted = (p >= threshold).astype(int)
    tp = int(((predicted == 1) & (y == 1)).sum()); fp = int(((predicted == 1) & (y == 0)).sum())
    fn = int(((predicted == 0) & (y == 1)).sum()); tn = int(((predicted == 0) & (y == 0)).sum())
    both = len(np.unique(y)) == 2
    return {
        "pr_auc": float(average_precision_score(y, p)) if y.sum() else np.nan,
        "roc_auc": float(roc_auc_score(y, p)) if both else np.nan,
        "precision": float(precision_score(y, predicted, zero_division=0)),
        "recall": float(recall_score(y, predicted, zero_division=0)),
        "f1": float(f1_score(y, predicted, zero_division=0)),
        "fpr": fp / max(fp + tn, 1),
        "alerts_per_attack": (tp + fp) / max(tp + fn, 1),
        "tp": tp, "fp": fp, "fn": fn, "tn": tn, "attack_edges": int(y.sum()), "edges": len(y),
    }


def multilabel_scores(Y, P, threshold):
    predicted = (P >= threshold).astype(int)
    present = Y.sum(0) > 0
    return {
        "micro_pr_auc": float(average_precision_score(Y[:, present], P[:, present], average="micro")) if present.any() else np.nan,
        "micro_precision": float(precision_score(Y, predicted, average="micro", zero_division=0)),
        "micro_recall": float(recall_score(Y, predicted, average="micro", zero_division=0)),
        "micro_f1": float(f1_score(Y, predicted, average="micro", zero_division=0)),
        "macro_f1": float(f1_score(Y[:, present], predicted[:, present], average="macro", zero_division=0)) if present.any() else np.nan,
        "edges": len(Y),
    }

Device: cpu


In [ ]:
def run_once(task, role, run_name):
    """Train on role == 'train', select on 'val', score 'test'. Returns metrics and test-edge predictions."""
    out_file = RESULT_DIR / f"{run_name}__{task}.parquet"
    metrics_file = RESULT_DIR / f"{run_name}__{task}.json"
    if out_file.exists() and metrics_file.exists():
        return json.load(open(metrics_file)), pd.read_parquet(out_file)

    torch.manual_seed(CFG["SEED"]); np.random.seed(CFG["SEED"])
    usable = TASK_LABELS[task]["usable"]
    graph_role = role[GRAPH_FIRST_ROW]
    pick = lambda name: [g for g, r, u in zip(graphs, graph_role, GRAPH_USABLE[task]) if r == name and u]
    mixed = sum(len(set(role[g.row_ids.numpy()])) > 1 for g in graphs)
    assert mixed == 0, f"{mixed} graphs mix roles; use GRAPH_SCOPE='capture' for this split"
    train_graphs, val_graphs, test_graphs = pick("train"), pick("val"), pick("test")
    if not (train_graphs and val_graphs and test_graphs):
        print(f"skipped {task} {run_name}: no labelled edges in train, val or test")
        return None, None

    ## scaler fitted on this run's training edges only
    train_rows = np.flatnonzero((role == "train") & usable)
    mean = torch.tensor(EDGE_FEATURES[train_rows].mean(0), device=device)
    std = torch.tensor(EDGE_FEATURES[train_rows].std(0), device=device).clamp_min(1e-6)

    if task == "binary":
        model = EdgeGraphSAGE(1).to(device)
        normal, attack = (BINARY_Y[train_rows] == 0).sum(), (BINARY_Y[train_rows] == 1).sum()
        loss_function = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(np.sqrt(normal / max(attack, 1)), dtype=torch.float32, device=device))
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    else:
        Y = TASK_LABELS[task]["Y"]
        model = EdgeGraphSAGE(Y.shape[1]).to(device)
        positive = torch.tensor(Y[train_rows].sum(0), device=device)
        pos_weight = (len(train_rows) - positive) / positive.clamp_min(1)
        loss_function = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)

    def forward(batch):
        batch = batch.to(device)
        batch.target_edge_attr = (batch.target_edge_attr - mean) / std
        rows = batch.row_ids.cpu().numpy()
        keep = usable[rows]
        logits = model(batch)[torch.tensor(keep, device=device)]
        if task == "binary":
            return logits.squeeze(-1), torch.tensor(BINARY_Y[rows[keep]], device=device), rows[keep]
        return logits, torch.tensor(TASK_LABELS[task]["Y"][rows[keep]], device=device), rows[keep]

    @torch.no_grad()
    def predict(graph_list):
        model.eval()
        probabilities, targets, row_ids = [], [], []
        for batch in DataLoader(graph_list, batch_size=128, shuffle=False):
            logits, target, rows = forward(batch)
            probabilities.append(torch.sigmoid(logits).cpu().numpy()); targets.append(target.cpu().numpy()); row_ids.append(rows)
        return np.concatenate(probabilities), np.concatenate(targets), np.concatenate(row_ids)

    def selection_score(p, y):
        if task == "binary":
            return average_precision_score(y, p) if y.sum() else 0.0
        present = y.sum(0) > 0
        return average_precision_score(y[:, present], p[:, present], average="micro") if present.any() else 0.0

    best_score, best_state, best_epoch, waiting = -1.0, None, 0, 0
    for epoch in range(1, CFG["EPOCHS"] + 1):
        model.train()
        for batch in DataLoader(train_graphs, batch_size=64, shuffle=True):
            logits, target, _ = forward(batch)
            if len(target) == 0:
                continue
            optimizer.zero_grad()
            loss = loss_function(logits, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
        score = selection_score(*predict(val_graphs)[:2])
        if score > best_score:
            best_score, best_epoch, waiting = score, epoch, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            waiting += 1
        if waiting >= CFG["PATIENCE"]:
            break
    model.load_state_dict(best_state)

    val_p, val_y, _ = predict(val_graphs)
    test_p, test_y, test_rows = predict(test_graphs)
    if task == "binary":
        precision, recall, thresholds = precision_recall_curve(val_y, val_p)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        threshold = float(thresholds[np.argmax(f1[:-1])]) if len(thresholds) else 0.5
        metrics = {"val": binary_scores(val_y, val_p, threshold), "test": binary_scores(test_y, test_p, threshold)}
        predictions = pd.DataFrame({"row_id": test_rows, "probability": test_p})
    else:
        grid = np.arange(0.10, 0.91, 0.05)
        threshold = float(max(grid, key=lambda t: f1_score(val_y, val_p >= t, average="micro", zero_division=0)))
        metrics = {"val": multilabel_scores(val_y, val_p, threshold), "test": multilabel_scores(test_y, test_p, threshold)}
        predictions = pd.DataFrame({"row_id": test_rows, "probability": list(test_p)})

    metrics.update({"run": run_name, "task": task, "threshold": threshold, "best_epoch": best_epoch,
                    "train_edges": int(len(train_rows)), "roles": pd.Series(role).value_counts().to_dict()})
    predictions["threshold"] = threshold
    predictions.to_parquet(out_file, index=False)
    json.dump(metrics, open(metrics_file, "w"), indent=1, default=float)
    return metrics, predictions

## 9. Runs
Each run is saved under `results/`; re-running the notebook skips finished runs, so a disconnected Colab session can resume.

In [ ]:
runs = []
for task in CFG["TASKS"]:
    if CFG["RUN_REFERENCE"]:
        runs.append((task, "reference", roles_reference))
    if CFG["RUN_CV"]:
        for repeat in CFG["REPEATS_TO_RUN"]:
            for k in range(CFG["N_FOLDS"]):
                runs.append((task, f"cv_r{repeat}_k{k}", lambda repeat=repeat, k=k: roles_cv(repeat, k)))
    if CFG["RUN_LOYO"]:
        for year_group in ["2021-22", "2024", "2025"]:
            runs.append((task, f"loyo_{year_group}", lambda year_group=year_group: roles_loyo(year_group)))

all_metrics = []
for task, run_name, make_role in runs:
    started = time.time()
    metrics, _ = run_once(task, make_role(), run_name)
    if metrics is None:
        continue
    all_metrics.append(metrics)
    test = metrics["test"]
    headline = test.get("pr_auc", test.get("micro_pr_auc"))
    print(f"{task:9s} {run_name:14s} test PR-AUC {headline:.4f} | best epoch {metrics['best_epoch']} | {(time.time() - started) / 60:.1f} min")

binary    reference      test PR-AUC 0.9229 | best epoch 4 | 0.0 min
binary    cv_r0_k0       test PR-AUC 0.8623 | best epoch 20 | 0.0 min
binary    cv_r0_k1       test PR-AUC 0.3599 | best epoch 2 | 0.0 min
binary    cv_r0_k2       test PR-AUC 0.6736 | best epoch 9 | 0.0 min
binary    cv_r0_k3       test PR-AUC 0.9765 | best epoch 20 | 0.0 min
binary    cv_r0_k4       test PR-AUC 0.7927 | best epoch 13 | 0.0 min
binary    cv_r1_k0       test PR-AUC 0.9670 | best epoch 14 | 0.9 min
binary    cv_r1_k1       test PR-AUC 0.9370 | best epoch 16 | 1.1 min
binary    cv_r1_k2       test PR-AUC 0.7095 | best epoch 11 | 0.7 min
binary    cv_r1_k3       test PR-AUC 0.9132 | best epoch 2 | 0.3 min
binary    cv_r1_k4       test PR-AUC 0.9989 | best epoch 9 | 0.5 min
binary    cv_r2_k0       test PR-AUC 0.8515 | best epoch 9 | 0.7 min
binary    cv_r2_k1       test PR-AUC 0.9746 | best epoch 12 | 0.8 min
binary    cv_r2_k2       test PR-AUC 1.0000 | best epoch 20 | 0.8 min
binary    cv_r2_k3       t

In [23]:
## 9b. Shortcut baselines on the same folds
## edge_only:   gradient boosting on the 10 edge features, no graph
## window_only: gradient boosting on window activity only (this window + the previous hour), nothing about the edge itself
from sklearn.ensemble import HistGradientBoostingClassifier

Y_ALL = edges.binary_target.to_numpy().astype(int)

def window_table():
    keys = ["capture"] if CFG["GRAPH_SCOPE"] == "capture" else []
    w = edges.groupby(keys + ["window_start"]).agg(
        win_edges=("src_ip_zeek", "size"),
        win_src=("src_ip_zeek", "nunique"),
        win_dst=("dest_ip_zeek", "nunique"),
        win_conn=("connection_count", "sum"),
        win_bytes=("total_orig_bytes", "sum"),
    ).reset_index()
    cols = ["win_edges", "win_src", "win_dst", "win_conn", "win_bytes"]
    parts = [w] if not keys else [g for _, g in w.groupby("capture")]
    out = []
    for g in parts:
        g = g.sort_values("window_start").set_index("window_start")
        context = g[cols].rolling("60min", closed="left").sum().add_prefix("ctx_").fillna(0)
        out.append(pd.concat([g, context], axis=1).reset_index())
    return pd.concat(out), keys

window_features, window_keys = window_table()
merged = edges[window_keys + ["window_start"]].merge(window_features, on=window_keys + ["window_start"], how="left")
WINDOW_X = np.log1p(merged.drop(columns=window_keys + ["window_start"]).to_numpy(np.float64))
BASELINE_X = {"edge_only": EDGE_FEATURES, "window_only": WINDOW_X}
print({name: X.shape for name, X in BASELINE_X.items()})

{'edge_only': (726307, 10), 'window_only': (726307, 10)}


In [24]:
def run_baseline(name, role, run_name):
    out_file = RESULT_DIR / f"{run_name}__{name}.parquet"
    metrics_file = RESULT_DIR / f"{run_name}__{name}.json"
    if out_file.exists() and metrics_file.exists():
        return json.load(open(metrics_file))

    X = BASELINE_X[name]
    train, val, test = (role == "train"), (role == "val"), (role == "test")
    model = HistGradientBoostingClassifier(max_iter=300, class_weight="balanced", random_state=CFG["SEED"])
    model.fit(X[train], Y_ALL[train])
    val_p = model.predict_proba(X[val])[:, 1]
    test_p = model.predict_proba(X[test])[:, 1]

    ## same threshold rule as GraphSAGE: best F1 on validation
    precision, recall, thresholds = precision_recall_curve(Y_ALL[val], val_p)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    threshold = float(thresholds[np.argmax(f1[:-1])]) if len(thresholds) else 0.5

    metrics = {"val": binary_scores(Y_ALL[val], val_p, threshold),
               "test": binary_scores(Y_ALL[test], test_p, threshold),
               "run": run_name, "task": name, "threshold": threshold, "best_epoch": 0}
    pd.DataFrame({"row_id": np.flatnonzero(test), "probability": test_p, "threshold": threshold}).to_parquet(out_file, index=False)
    json.dump(metrics, open(metrics_file, "w"), indent=1, default=float)
    return metrics


baseline_runs = [("reference", roles_reference)]
for repeat in CFG["REPEATS_TO_RUN"]:
    for k in range(CFG["N_FOLDS"]):
        baseline_runs.append((f"cv_r{repeat}_k{k}", lambda repeat=repeat, k=k: roles_cv(repeat, k)))
for year_group in ["2021-22", "2024", "2025"]:
    baseline_runs.append((f"loyo_{year_group}", lambda year_group=year_group: roles_loyo(year_group)))

for run_name, make_role in baseline_runs:
    role = make_role()
    for name in BASELINE_X:
        m = run_baseline(name, role, run_name)
        print(f"{name:11s} {run_name:14s} test PR-AUC {m['test']['pr_auc']:.4f}")

edge_only   reference      test PR-AUC 0.1733
window_only reference      test PR-AUC 1.0000
edge_only   cv_r0_k0       test PR-AUC 0.8769
window_only cv_r0_k0       test PR-AUC 1.0000
edge_only   cv_r0_k1       test PR-AUC 0.1011
window_only cv_r0_k1       test PR-AUC 1.0000
edge_only   cv_r0_k2       test PR-AUC 0.9583
window_only cv_r0_k2       test PR-AUC 1.0000
edge_only   cv_r0_k3       test PR-AUC 0.8490
window_only cv_r0_k3       test PR-AUC 0.5700
edge_only   cv_r0_k4       test PR-AUC 0.4538
window_only cv_r0_k4       test PR-AUC 1.0000
edge_only   loyo_2021-22   test PR-AUC 0.4220
window_only loyo_2021-22   test PR-AUC 0.0049
edge_only   loyo_2024      test PR-AUC 0.6315
window_only loyo_2024      test PR-AUC 0.5650
edge_only   loyo_2025      test PR-AUC 0.9167
window_only loyo_2025      test PR-AUC 1.0000


In [29]:
def load_metrics(task, run_name):
    path = RESULT_DIR / f"{run_name}__{task}.json"
    return json.load(open(path))["test"] if path.exists() else None

def pooled(task, repeat):
    parts = []
    for k in range(CFG["N_FOLDS"]):
        path = RESULT_DIR / f"cv_r{repeat}_k{k}__{task}.parquet"
        if path.exists():
            parts.append(pd.read_parquet(path))
        else:
            print(f"Warning: Missing fold file {path.name}")
    if not parts:
        return None
    oof = pd.concat(parts).set_index("row_id").join(edges[["binary_target", "capture"]])
    oof["predicted"] = (oof.probability >= oof.threshold).astype(int)
    return oof

rows = []
for task in ["binary", "edge_only", "window_only"]:
    for repeat in CFG["REPEATS_TO_RUN"]:
        oof = pooled(task, repeat)
        if oof is None:
            continue
        y, p, d = oof.binary_target.to_numpy(), oof.probability.to_numpy(), oof.predicted.to_numpy()
        per_capture = oof.groupby("capture").apply(
            lambda g: average_precision_score(g.binary_target, g.probability), include_groups=False)
        counts = binary_scores(y, d.astype(float), 0.5)
        rows.append({
            "model": "GraphSAGE" if task == "binary" else task, "repeat": repeat,
            "cv_pooled_pr_auc": average_precision_score(y, p),
            "cv_capture_avg_pr_auc": per_capture.mean(),
            "cv_recall": counts["recall"], "cv_precision": counts["precision"], "cv_fpr": counts["fpr"],
            "reference_pr_auc": (load_metrics(task, "reference") or {}).get("pr_auc"),
            **{f"loyo_{y_}_pr_auc": (load_metrics(task, f"loyo_{y_}") or {}).get("pr_auc")
               for y_ in ["2021-22", "2024", "2025"]},
        })
if rows:
    comparison = pd.DataFrame(rows).set_index(["model", "repeat"]).round(4)
    comparison.to_csv(OUT_DIR / f"baseline_comparison_{RESULT_DIR.name.replace('results_', '')}.csv")
    display(comparison)
else:
    print("No matching evaluation metrics found. Please ensure your cross-validation folds have finished running.")

,,cv_pooled_pr_auc,cv_capture_avg_pr_auc,cv_recall,cv_precision,cv_fpr,reference_pr_auc,loyo_2021-22_pr_auc,loyo_2024_pr_auc,loyo_2025_pr_auc
model,repeat,,,,,,,,,
edge_only,0,0.6494,0.7624,0.2083,0.757,0.0054,0.1733,0.4220,0.6315,0.9167
window_only,0,0.4435,0.9516,0.0591,0.889,0.0006,1.0000,0.0049,0.5650,1.0000


In [31]:
## 9c. Leave-one-capture-out: train on five captures, test on the sixth
import os
import sys
import json
import time
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import average_precision_score, precision_recall_curve, f1_score

CAPTURES_ALL = sorted(edges.capture.unique())
all_metrics = []  # Initialize to prevent NameError if cell 9 is skipped or kernel restarted

def roles_loco(capture):
    role = np.where(edges.capture == capture, "test", "train").astype(object)
    rest = np.flatnonzero(role == "train")
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=CFG["SEED"])
    _, val_index = next(splitter.split(rest, edges.stratum.to_numpy()[rest], groups=edges.block_id.to_numpy()[rest]))
    role[rest[val_index]] = "val"
    ## the whole test capture is held out, so only train windows next to validation windows need purging
    role[(role == "train") & near(WINDOW_TIMES, WINDOW_TIMES[role == "val"], PURGE)] = "purged"
    return role

# Redefining run_once to prevent NameError in case of kernel restarts
def run_once(task, role, run_name):
    out_file = RESULT_DIR / f"{run_name}__{task}.parquet"
    metrics_file = RESULT_DIR / f"{run_name}__{task}.json"
    if out_file.exists() and metrics_file.exists():
        return json.load(open(metrics_file)), pd.read_parquet(out_file)

    torch.manual_seed(CFG["SEED"]); np.random.seed(CFG["SEED"])
    usable = TASK_LABELS[task]["usable"]
    graph_role = role[GRAPH_FIRST_ROW]
    pick = lambda name: [g for g, r, u in zip(graphs, graph_role, GRAPH_USABLE[task]) if r == name and u]
    train_graphs, val_graphs, test_graphs = pick("train"), pick("val"), pick("test")
    if not (train_graphs and val_graphs and test_graphs):
        print(f"skipped {task} {run_name}: no labelled edges in train, val or test")
        return None, None

    train_rows = np.flatnonzero((role == "train") & usable)
    mean = torch.tensor(EDGE_FEATURES[train_rows].mean(0), device=device)
    std = torch.tensor(EDGE_FEATURES[train_rows].std(0), device=device).clamp_min(1e-6)

    if task == "binary":
        model = EdgeGraphSAGE(1).to(device)
        normal, attack = (BINARY_Y[train_rows] == 0).sum(), (BINARY_Y[train_rows] == 1).sum()
        loss_function = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(np.sqrt(normal / max(attack, 1)), dtype=torch.float32, device=device))
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    else:
        Y = TASK_LABELS[task]["Y"]
        model = EdgeGraphSAGE(Y.shape[1]).to(device)
        positive = torch.tensor(Y[train_rows].sum(0), device=device)
        pos_weight = (len(train_rows) - positive) / positive.clamp_min(1)
        loss_function = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)

    def forward(batch):
        batch = batch.to(device)
        batch.target_edge_attr = (batch.target_edge_attr - mean) / std
        rows = batch.row_ids.cpu().numpy()
        keep = usable[rows]
        logits = model(batch)[torch.tensor(keep, device=device)]
        if task == "binary":
            return logits.squeeze(-1), torch.tensor(BINARY_Y[rows[keep]], device=device), rows[keep]
        return logits, torch.tensor(TASK_LABELS[task]["Y"][rows[keep]], device=device), rows[keep]

    @torch.no_grad()
    def predict(graph_list):
        model.eval()
        probabilities, targets, row_ids = [], [], []
        for batch in DataLoader(graph_list, batch_size=128, shuffle=False):
            logits, target, rows = forward(batch)
            probabilities.append(torch.sigmoid(logits).cpu().numpy()); targets.append(target.cpu().numpy()); row_ids.append(rows)
        return np.concatenate(probabilities), np.concatenate(targets), np.concatenate(row_ids)

    best_score, best_state, best_epoch, waiting = -1.0, None, 0, 0
    for epoch in range(1, CFG["EPOCHS"] + 1):
        model.train()
        for batch in DataLoader(train_graphs, batch_size=64, shuffle=True):
            logits, target, _ = forward(batch)
            if len(target) == 0: continue
            optimizer.zero_grad()
            loss = loss_function(logits, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

        val_p, val_y, _ = predict(val_graphs)
        score = average_precision_score(val_y, val_p) if val_y.sum() else 0.0
        if score > best_score:
            best_score, best_epoch, waiting = score, epoch, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            waiting += 1
        if waiting >= CFG["PATIENCE"]: break

    model.load_state_dict(best_state)
    val_p, val_y, _ = predict(val_graphs)
    test_p, test_y, test_rows = predict(test_graphs)
    precision, recall, thresholds = precision_recall_curve(val_y, val_p)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    threshold = float(thresholds[np.argmax(f1[:-1])]) if len(thresholds) else 0.5
    metrics = {"val": binary_scores(val_y, val_p, threshold), "test": binary_scores(test_y, test_p, threshold)}
    predictions = pd.DataFrame({"row_id": test_rows, "probability": test_p})
    metrics.update({"run": run_name, "task": task, "threshold": threshold, "best_epoch": best_epoch, "train_edges": int(len(train_rows))})
    predictions["threshold"] = threshold
    predictions.to_parquet(out_file, index=False)
    json.dump(metrics, open(metrics_file, "w"), indent=1, default=float)
    return metrics, predictions

for capture in CAPTURES_ALL:
    role = roles_loco(capture)
    run_name = f"loco_{capture}"
    metrics, _ = run_once("binary", role, run_name)
    if metrics is not None:
        print(f"{capture:22s} test PR-AUC {metrics['test']['pr_auc']:.4f}")

UWF-ZeekData22         test PR-AUC 0.5903
UWF-ZeekData24         test PR-AUC 0.9597
UWF-ZeekDataFall22     test PR-AUC 0.5962
UWF-ZeekDataFall24-2   test PR-AUC 0.1159
UWF-ZeekDataSum25-1    test PR-AUC 0.9435
UWF-ZeekDataSum25-2    test PR-AUC 0.8919


In [ ]:
loco_rows, per_capture_rows = [], []
for task in ["binary", "edge_only", "window_only"]:
    model = "GraphSAGE" if task == "binary" else task
    parts = [pd.read_parquet(RESULT_DIR / f"loco_{c}__{task}.parquet") for c in CAPTURES_ALL]
    oof = pd.concat(parts).set_index("row_id").join(edges[["binary_target", "capture"]])
    oof["predicted"] = (oof.probability >= oof.threshold).astype(int)
    y, p = oof.binary_target.to_numpy(), oof.probability.to_numpy()
    counts = binary_scores(y, oof.predicted.to_numpy().astype(float), 0.5)
    for capture, g in oof.groupby("capture"):
        s = binary_scores(g.binary_target.to_numpy(), g.predicted.to_numpy().astype(float), 0.5)
        per_capture_rows.append({"model": model, "capture": capture,
                                 "pr_auc": average_precision_score(g.binary_target, g.probability),
                                 "recall": s["recall"], "precision": s["precision"], "fpr": s["fpr"]})
    capture_pr = pd.DataFrame([r for r in per_capture_rows if r["model"] == model]).pr_auc
    loco_rows.append({"model": model, "edges_tested": len(oof), "pooled_pr_auc": average_precision_score(y, p),
                      "capture_avg_pr_auc": capture_pr.mean(), "recall": counts["recall"],
                      "precision": counts["precision"], "fpr": counts["fpr"]})

loco_summary = pd.DataFrame(loco_rows).set_index("model").round(4)
loco_by_capture = pd.DataFrame(per_capture_rows).pivot(index="capture", columns="model", values="pr_auc").round(4)
loco_summary.to_csv(OUT_DIR / "loco_summary.csv"); loco_by_capture.to_csv(OUT_DIR / "loco_by_capture.csv")
display(loco_summary); display(loco_by_capture)

,edges_tested,pooled_pr_auc,capture_avg_pr_auc,recall,precision,fpr
model,,,,,,
GraphSAGE,726307,0.8881,0.6829,0.8265,0.7705,0.0200
edge_only,726307,0.5856,0.5934,0.6123,0.3851,0.0795
window_only,726307,0.1089,0.7789,0.0578,0.0396,0.1142


model,GraphSAGE,edge_only,window_only
capture,,,
UWF-ZeekData22,0.5903,0.0414,0.8330
UWF-ZeekData24,0.9597,0.8208,0.6161
UWF-ZeekDataFall22,0.5962,0.3586,0.2241
UWF-ZeekDataFall24-2,0.1159,0.4655,1.0000
UWF-ZeekDataSum25-1,0.9435,0.8989,1.0000
UWF-ZeekDataSum25-2,0.8919,0.9755,1.0000


In [32]:
CFG["REPEATS_TO_RUN"] = [0, 1, 2]
print(CFG["MAX_BLOCK_HOURS"], RESULT_DIR)   # check: None, and .../results_global_maxnoneh

None /content/drive/Shareddrives/298B/blocked_cv/results_capture_maxnoneh


## 10. Results
Per-run test metrics, mean ± std over folds, and pooled out-of-fold scores (every edge scored once per repeat).

In [34]:
def metrics_table(task):
    rows = []
    for m in all_metrics:
        if m["task"] == task:
            rows.append({"run": m["run"], "threshold": m["threshold"], "best_epoch": m["best_epoch"], **m["test"]})
    if not rows:
        return pd.DataFrame(columns=["run"]).set_index("run")
    return pd.DataFrame(rows).set_index("run")


for task in CFG["TASKS"]:
    table = metrics_table(task)
    print(f"\n=== {task}: test metrics per run ===")
    if table.empty:
        print("No metrics found. Ensure training runs in cell 9 have been executed.")
    else:
        print(table.round(4).to_string())
        cv = table[table.index.str.startswith("cv_")]
        if len(cv):
            print(f"\n{task}: cross-validation mean ± std over {len(cv)} folds")
            print(pd.DataFrame({"mean": cv.mean(numeric_only=True), "std": cv.std(numeric_only=True)}).round(4).to_string())


=== binary: test metrics per run ===
No metrics found. Ensure training runs in cell 9 have been executed.


In [ ]:
## pooled out-of-fold results for the binary detector, per repeat, with breakdowns
def pooled_binary(repeat):
    parts = [pd.read_parquet(RESULT_DIR / f"cv_r{repeat}_k{k}__binary.parquet").assign(fold=k) for k in range(CFG["N_FOLDS"])]
    oof = pd.concat(parts).set_index("row_id").join(edges[["binary_target", "year_group", "capture", "technique"]])
    oof["predicted"] = (oof.probability >= oof.threshold).astype(int)
    return oof


if "binary" in CFG["TASKS"] and CFG["RUN_CV"]:
    for repeat in CFG["REPEATS_TO_RUN"]:
        oof = pooled_binary(repeat)
        y, p, predicted = oof.binary_target.to_numpy(), oof.probability.to_numpy(), oof.predicted.to_numpy()
        ## thresholded counts use each fold's own validation threshold; PR-AUC / ROC-AUC use the raw probabilities
        pooled = binary_scores(y, predicted.astype(float), 0.5)
        pooled["pr_auc"] = float(average_precision_score(y, p))
        pooled["roc_auc"] = float(roc_auc_score(y, p))
        print(f"\n=== repeat {repeat}: pooled out-of-fold scores for {len(oof):,} of {len(edges):,} edges ===")
        print(pd.Series(pooled).round(4).to_string())

        def group_scores(frame):
            yy, pp, dd = frame.binary_target.to_numpy(), frame.probability.to_numpy(), frame.predicted.to_numpy()
            s = binary_scores(yy, dd.astype(float), 0.5)
            return pd.Series({"edges": len(frame), "attack_edges": int(yy.sum()),
                              "pr_auc": average_precision_score(yy, pp) if yy.sum() else np.nan,
                              "recall": s["recall"], "precision": s["precision"], "fpr": s["fpr"]})

        print("\nby year group"); print(oof.groupby("year_group").apply(group_scores).round(4).to_string())
        print("\nby capture"); print(oof.groupby("capture").apply(group_scores).round(4).to_string())
        by_technique = oof[oof.binary_target == 1].groupby("technique").agg(attack_edges=("predicted", "size"), recall=("predicted", "mean"))
        by_technique = by_technique.join(coverage[["blocks", "insufficient_data"]])
        print("\nrecall by technique (attack edges only)"); print(by_technique.sort_values("attack_edges", ascending=False).round(4).to_string())


=== repeat 0: pooled out-of-fold scores for 726,307 of 726,307 edges ===
pr_auc                    0.9383
roc_auc                   0.9848
precision                 0.9388
recall                    0.6867
f1                        0.7932
fpr                       0.0036
alerts_per_attack         0.7315
tp                    37529.0000
fp                     2445.0000
fn                    17121.0000
tn                   669212.0000
attack_edges          54650.0000
edges                726307.0000

by year group


/tmp/ipykernel_2865/2617061815.py:27: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print("\nby year group"); print(oof.groupby("year_group").apply(group_scores).round(4).to_string())


               edges  attack_edges  pr_auc  recall  precision     fpr
year_group                                                           
2021-22     178486.0         779.0  0.6543  0.7946     0.7042  0.0015
2024        295052.0       52913.0  0.9537  0.6804     0.9572  0.0066
2025        252769.0         958.0  0.9533  0.9468     0.6116  0.0023

by capture


/tmp/ipykernel_2865/2617061815.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print("\nby capture"); print(oof.groupby("capture").apply(group_scores).round(4).to_string())


                         edges  attack_edges  pr_auc  recall  precision     fpr
capture                                                                        
UWF-ZeekData22        145608.0         340.0  0.6949  0.9147     0.5495  0.0018
UWF-ZeekData24        109011.0       51771.0  0.9719  0.6834     0.9573  0.0276
UWF-ZeekDataFall22     32878.0         439.0  0.7945  0.7016     0.9840  0.0002
UWF-ZeekDataFall24-2  186041.0        1142.0  0.5966  0.5464     0.9512  0.0002
UWF-ZeekDataSum25-1   179554.0         675.0  0.9479  0.9319     0.6545  0.0019
UWF-ZeekDataSum25-2    73215.0         283.0  0.9017  0.9823     0.5326  0.0033

recall by technique (attack edges only)
                                  attack_edges  recall  blocks  insufficient_data
technique                                                                        
T1595                                    37156  0.6007      30              False
T1078                                     5426  0.8056       3           

...

/tmp/ipykernel_2865/2617061815.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print("\nby capture"); print(oof.groupby("capture").apply(group_scores).round(4).to_string())


                         edges  attack_edges  pr_auc  recall  precision     fpr
capture                                                                        
UWF-ZeekData22        145608.0         340.0  0.5972  0.9618     0.4985  0.0023
UWF-ZeekData24        109011.0       51771.0  0.9938  0.9618     0.9678  0.0289
UWF-ZeekDataFall22     32878.0         439.0  0.7522  0.2642     0.7532  0.0012
UWF-ZeekDataFall24-2  186041.0        1142.0  0.6143  0.5403     0.5609  0.0026
UWF-ZeekDataSum25-1   179554.0         675.0  0.9453  0.9170     0.6045  0.0023
UWF-ZeekDataSum25-2    73215.0         283.0  0.9875  0.1166     0.5156  0.0004

recall by technique (attack edges only)
                                  attack_edges  recall  blocks  insufficient_data
technique                                                                        
T1595                                    37156  0.9462      30              False
T1078                                     5426  0.9910       3           

/tmp/ipykernel_2865/2617061815.py:27: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print("\nby year group"); print(oof.groupby("year_group").apply(group_scores).round(4).to_string())


               edges  attack_edges  pr_auc  recall  precision     fpr
year_group                                                           
2021-22     178486.0         779.0  0.6034  0.7561     0.7895  0.0009
2024        295052.0       52913.0  0.9996  0.9825     0.9945  0.0012
2025        252769.0         958.0  0.9417  0.6837     0.8827  0.0003

by capture
                         edges  attack_edges  pr_auc  recall  precision     fpr
capture                                                                        
UWF-ZeekData22        145608.0         340.0  0.6107  0.9529     0.6894  0.0010
UWF-ZeekData24        109011.0       51771.0  0.9998  0.9876     0.9945  0.0050
UWF-ZeekDataFall22     32878.0         439.0  0.7783  0.6036     0.9601  0.0003
UWF-ZeekDataFall24-2  186041.0        1142.0  0.9668  0.7513     0.9942  0.0000
UWF-ZeekDataSum25-1   179554.0         675.0  0.9472  0.9259     0.9455  0.0002
UWF-ZeekDataSum25-2    73215.0         283.0  0.8068  0.1060     0.3704  0.000

/tmp/ipykernel_2865/2617061815.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print("\nby capture"); print(oof.groupby("capture").apply(group_scores).round(4).to_string())


In [ ]:
summary = {task: metrics_table(task).round(4).reset_index().to_dict("records") for task in CFG["TASKS"]}
summary_file = OUT_DIR / f"summary_{CFG['GRAPH_SCOPE']}.json"
json.dump({"config": CFG, "runs": summary}, open(summary_file, "w"), indent=1, default=str)
print("Saved:", summary_file)

Saved: /content/drive/Shareddrives/298B/blocked_cv/summary_global.json


In [6]:
for seed in [43, 44]:
    CFG["SEED"] = seed                          # training seed
    for capture in CAPTURES_ALL:
        CFG["SEED"] = 42; role = roles_loco(capture); CFG["SEED"] = seed   # same validation blocks
        metrics, _ = run_once("binary", role, f"loco_{capture}_s{seed}")
        print(f"seed {seed} {capture:22s} PR-AUC {metrics['test']['pr_auc']:.4f}")
CFG["SEED"] = 42

NameError: name 'CFG' is not defined

In [35]:
rows = []
for c in CAPTURES_ALL:
    pred = pd.read_parquet(RESULT_DIR / f"loco_{c}__binary.parquet").set_index("row_id")
    test_attacks = edges.loc[pred.index].query("binary_target == 1")
    train_attackers = set(edges[(edges.capture != c) & (edges.binary_target == 1)].src_ip_zeek)
    familiar = test_attacks.src_ip_zeek.isin(train_attackers)
    caught = pred.loc[test_attacks.index, "probability"] >= pred.loc[test_attacks.index, "threshold"]
    rows.append({"recording_tested": c,
                 "attacks_familiar_attacker": int(familiar.sum()),
                 "caught_familiar": caught[familiar].mean() if familiar.any() else None,
                 "attacks_new_attacker": int((~familiar).sum()),
                 "caught_new": caught[~familiar].mean() if (~familiar).any() else None})
display(pd.DataFrame(rows).round(3))

,recording_tested,attacks_familiar_attacker,caught_familiar,attacks_new_attacker,caught_new
0,UWF-ZeekData22,283,0.968,57,0.895
1,UWF-ZeekData24,14325,0.829,37446,0.834
2,UWF-ZeekDataFall22,393,0.504,46,0.543
3,UWF-ZeekDataFall24-2,854,0.406,288,0.986
4,UWF-ZeekDataSum25-1,23,0.913,652,0.883
5,UWF-ZeekDataSum25-2,7,1.000,276,1.000


In [36]:
att = edges[edges.binary_target == 1]
caps = sorted(edges.capture.unique())
rows = []
for i, a in enumerate(caps):
    for b in caps[i + 1:]:
        A, B = att[att.capture == a], att[att.capture == b]
        same_time = set(A.window_start) & set(B.window_start)
        rows.append({"recording_a": a, "recording_b": b,
                     "attack_slices_in_both": len(same_time),
                     "a_attacks_at_those_times": int(A.window_start.isin(same_time).sum()),
                     "b_attacks_at_those_times": int(B.window_start.isin(same_time).sum()),
                     "attacker_addresses_in_both": len(set(A.src_ip_zeek) & set(B.src_ip_zeek))})
display(pd.DataFrame(rows))

,recording_a,recording_b,attack_slices_in_both,a_attacks_at_those_times,b_attacks_at_those_times,attacker_addresses_in_both
0,UWF-ZeekData22,UWF-ZeekData24,0,0,0,1
1,UWF-ZeekData22,UWF-ZeekDataFall22,0,0,0,3
2,UWF-ZeekData22,UWF-ZeekDataFall24-2,0,0,0,2
3,UWF-ZeekData22,UWF-ZeekDataSum25-1,0,0,0,1
4,UWF-ZeekData22,UWF-ZeekDataSum25-2,0,0,0,1
5,UWF-ZeekData24,UWF-ZeekDataFall22,0,0,0,4
6,UWF-ZeekData24,UWF-ZeekDataFall24-2,0,0,0,1
7,UWF-ZeekData24,UWF-ZeekDataSum25-1,0,0,0,1
8,UWF-ZeekData24,UWF-ZeekDataSum25-2,0,0,0,0
9,UWF-ZeekDataFall22,UWF-ZeekDataFall24-2,0,0,0,3


In [37]:
## 9d. Rescale features against each recording's own traffic (no labels used)
CAP_ID = {c: i for i, c in enumerate(CAPTURES_ALL)}
edge_cap = edges.capture.map(CAP_ID).to_numpy()
graph_cap = edge_cap[GRAPH_FIRST_ROW]
assert all(len(set(edge_cap[g.row_ids.numpy()])) == 1 for g in graphs), "needs GRAPH_SCOPE='capture'"

def robust_scale(values, groups):
    values = values.astype(np.float64)
    out = np.empty_like(values)
    for grp in np.unique(groups):
        m = groups == grp
        median = np.median(values[m], axis=0)
        q75, q25 = np.percentile(values[m], [75, 25], axis=0)
        spread = np.where(q75 - q25 > 0, q75 - q25, values[m].std(0) + 1e-6)
        out[m] = (values[m] - median) / spread
    return out.astype(np.float32)

EDGE_FEATURES_RAW, GRAPHS_RAW, RESULT_DIR_RAW = EDGE_FEATURES, graphs, RESULT_DIR
EDGE_FEATURES_NORM = robust_scale(EDGE_FEATURES_RAW, edge_cap)

node_values = np.concatenate([g.x.numpy() for g in GRAPHS_RAW])
node_groups = np.concatenate([np.full(g.num_nodes, c) for g, c in zip(GRAPHS_RAW, graph_cap)])
node_norm = robust_scale(node_values, node_groups)

GRAPHS_NORM, start = [], 0
for g in GRAPHS_RAW:
    h = g.clone()
    h.x = torch.tensor(node_norm[start:start + g.num_nodes]); start += g.num_nodes
    h.target_edge_attr = torch.tensor(EDGE_FEATURES_NORM[g.row_ids.numpy()])
    GRAPHS_NORM.append(h)
print("rescaled graphs:", len(GRAPHS_NORM))

rescaled graphs: 33607


In [39]:
NORM_DIR = OUT_DIR / "results_capture_maxnoneh_norm"
NORM_DIR.mkdir(exist_ok=True)
EDGE_FEATURES, graphs, RESULT_DIR = EDGE_FEATURES_NORM, GRAPHS_NORM, NORM_DIR
try:
    started = time.time()
    metrics, _ = run_once("binary", roles_loco("UWF-ZeekDataFall24-2"), "loco_UWF-ZeekDataFall24-2")
    print(f"Fall24-2 PR-AUC {metrics['test']['pr_auc']:.4f} (before: 0.1159) | {(time.time() - started) / 60:.1f} min")
finally:
    EDGE_FEATURES, graphs, RESULT_DIR = EDGE_FEATURES_RAW, GRAPHS_RAW, RESULT_DIR_RAW

Fall24-2 PR-AUC 0.5439 (before: 0.1159) | 2.7 min


In [40]:
NORM_DIR = OUT_DIR / "results_capture_maxnoneh_norm"
NORM_DIR.mkdir(exist_ok=True)
EDGE_FEATURES, graphs, RESULT_DIR = EDGE_FEATURES_NORM, GRAPHS_NORM, NORM_DIR
try:
    for capture in CAPTURES_ALL:
        metrics, _ = run_once("binary", roles_loco(capture), f"loco_{capture}")
        print(f"{capture:22s} PR-AUC {metrics['test']['pr_auc']:.4f}")
finally:
    EDGE_FEATURES, graphs, RESULT_DIR = EDGE_FEATURES_RAW, GRAPHS_RAW, RESULT_DIR_RAW   # always switch back

UWF-ZeekData22         PR-AUC 0.0816
UWF-ZeekData24         PR-AUC 0.3618
UWF-ZeekDataFall22     PR-AUC 0.5802
UWF-ZeekDataFall24-2   PR-AUC 0.5439
UWF-ZeekDataSum25-1    PR-AUC 0.0180
UWF-ZeekDataSum25-2    PR-AUC 0.9878


In [41]:
rows = []
for c in CAPTURES_ALL:
    rows.append({"recording": c,
                 "before": json.load(open(RESULT_DIR_RAW / f"loco_{c}__binary.json"))["test"]["pr_auc"],
                 "after": json.load(open(NORM_DIR / f"loco_{c}__binary.json"))["test"]["pr_auc"]})
for label, folder in [("before", RESULT_DIR_RAW), ("after", NORM_DIR)]:
    oof = pd.concat([pd.read_parquet(folder / f"loco_{c}__binary.parquet") for c in CAPTURES_ALL]).set_index("row_id").join(edges[["binary_target"]])
    s = binary_scores(oof.binary_target.to_numpy(), (oof.probability >= oof.threshold).astype(float).to_numpy(), 0.5)
    rows.append({"recording": f"ALL ({label})", label: average_precision_score(oof.binary_target, oof.probability),
                 "recall": s["recall"], "precision": s["precision"], "fpr": s["fpr"]})
display(pd.DataFrame(rows).round(4))

,recording,before,after,recall,precision,fpr
0,UWF-ZeekData22,0.5903,0.0816,NaN,NaN,NaN
1,UWF-ZeekData24,0.9597,0.3618,NaN,NaN,NaN
2,UWF-ZeekDataFall22,0.5962,0.5802,NaN,NaN,NaN
3,UWF-ZeekDataFall24-2,0.1159,0.5439,NaN,NaN,NaN
4,UWF-ZeekDataSum25-1,0.9435,0.0180,NaN,NaN,NaN
5,UWF-ZeekDataSum25-2,0.8919,0.9878,NaN,NaN,NaN
6,ALL (before),0.8881,NaN,0.8265,0.7705,0.0200
7,ALL (after),NaN,0.0438,0.0411,0.0290,0.1122
